In [24]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,accuracy_score
from sklearn.preprocessing import  OneHotEncoder,StandardScaler,LabelEncoder,OrdinalEncoder,RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer

In [25]:
df=pd.read_csv("loan_approval_dataset.csv")

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Age                1000 non-null   int64 
 1   Salary             1000 non-null   int64 
 2   Credit_Score       1000 non-null   int64 
 3   Loan_Amount        1000 non-null   int64 
 4   Loan_Term          1000 non-null   object
 5   Employment_Status  1000 non-null   object
 6   Residence_Type     1000 non-null   object
 7   Previous_Default   1000 non-null   object
 8   Loan_Approved      1000 non-null   object
dtypes: int64(4), object(5)
memory usage: 70.4+ KB


In [27]:
df.head(5)

,Age,Salary,Credit_Score,Loan_Amount,Loan_Term,Employment_Status,Residence_Type,Previous_Default,Loan_Approved
0,56,136748,584,38209,36 months,Employed,Owned,Yes,Yes
1,46,25287,815,27424,24 months,Self-Employed,Rented,No,Yes
2,32,146593,398,42396,12 months,Unemployed,Rented,Yes,Yes
3,60,54387,696,11370,24 months,Unemployed,Owned,No,No
4,25,28512,788,14528,12 months,Employed,Owned,No,No


In [28]:
x=df.drop('Loan_Approved',axis=1)
y=df.Loan_Approved

In [29]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [30]:
num_col=xtrain.select_dtypes(include="number").columns
obj_col=xtrain.select_dtypes(exclude="number").columns

In [31]:
def num_encoded_concat(df):
    num_col=xtrain.select_dtypes(include="number")
    obj_col=xtrain.select_dtypes(exclude="number")
    encode=OneHotEncoder(sparse_output=False)
    obj_encoded_cols=encode.fit_transform(obj_col)
    new_obj_cols=pd.DataFrame(obj_encoded_cols,columns=encode.get_feature_names_out())
    new_df=pd.concat([num_col.reset_index(drop=True),new_obj_cols.reset_index(drop=True)],axis=1)
    return new_df

In [32]:
xtrain=num_encoded_concat(xtrain)
xtest=num_encoded_concat(xtest)

In [33]:
xtrain

,Age,Salary,Credit_Score,Loan_Amount,Loan_Term_12 months,Loan_Term_24 months,Loan_Term_36 months,Loan_Term_48 months,Employment_Status_Employed,Employment_Status_Self-Employed,Employment_Status_Unemployed,Residence_Type_Mortgage,Residence_Type_Owned,Residence_Type_Rented,Previous_Default_No,Previous_Default_Yes
0,44,85441,507,17109,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
1,38,87298,489,10776,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,45,49629,762,39441,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,47,75337,809,40596,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,46,101121,504,31956,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,52,118723,689,31649,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
796,26,135654,388,22514,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
797,64,135005,683,8642,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
798,26,47856,841,45697,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


In [34]:
ytrain

29      No
535     No
695     No
557     No
836    Yes
      ... 
106     No
270     No
860     No
435    Yes
102    Yes
Name: Loan_Approved, Length: 800, dtype: object

**Without using pipeline**

In [35]:
grid_search_cv=GridSearchCV(
    estimator=LogisticRegression(),
    param_grid={'C':[0.01,0.1,1.0,10],
    'penalty':['l1','l2'],
    'solver':['liblinear'],
    'class_weight':['balanced',None]},

    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring='f1_macro'
)


In [36]:
grid_search_cv.fit(xtrain,ytrain)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,estimator,LogisticRegression()
,param_grid,"{'C': [0.01, 0.1, ...], 'class_weight': ['balanced', None], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l1'


In [37]:
grid_search_cv.best_params_

{'C': 0.01, 'class_weight': None, 'penalty': 'l1', 'solver': 'liblinear'}

In [38]:
cv_result=pd.DataFrame(grid_search_cv.cv_results_)

In [39]:
cv_result.sort_values(by='rank_test_score').reset_index(drop=True)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_class_weight,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.013585,0.016382,0.013985,0.012144,0.01,None,l1,liblinear,"{'C': 0.01, 'class_weight': None, 'penalty': '...",0.490386,0.474261,0.506076,0.503905,0.493572,0.493640,0.011368,1
1,0.009939,0.000762,0.006359,0.000661,1.00,balanced,l1,liblinear,"{'C': 1.0, 'class_weight': 'balanced', 'penalt...",0.524926,0.493730,0.543732,0.461153,0.436708,0.492050,0.039426,2
2,0.013764,0.003422,0.006943,0.001141,10.00,None,l1,liblinear,"{'C': 10, 'class_weight': None, 'penalty': 'l1...",0.542875,0.505767,0.524926,0.455207,0.428549,0.491465,0.042989,3
3,0.007718,0.001074,0.006849,0.001000,1.00,None,l1,liblinear,"{'C': 1.0, 'class_weight': None, 'penalty': 'l...",0.524926,0.493572,0.536849,0.467731,0.428549,0.490325,0.039240,4
4,0.006518,0.000862,0.010727,0.008319,0.01,balanced,l1,liblinear,"{'C': 0.01, 'class_weight': 'balanced', 'penal...",0.487179,0.455718,0.505303,0.493591,0.498747,0.488108,0.017256,5
5,0.013511,0.004418,0.006499,0.001117,10.00,balanced,l1,liblinear,"{'C': 10, 'class_weight': 'balanced', 'penalty...",0.524703,0.500000,0.524926,0.454524,0.423559,0.485542,0.040244,6
6,0.014127,0.009773,0.009439,0.001193,0.10,balanced,l2,liblinear,"{'C': 0.1, 'class_weight': 'balanced', 'penalt...",0.480743,0.456059,0.523137,0.440053,0.492143,0.478427,0.028863,7
7,0.007487,0.001997,0.007205,0.001728,10.00,balanced,l2,liblinear,"{'C': 10, 'class_weight': 'balanced', 'penalty...",0.480743,0.456059,0.523137,0.440053,0.492143,0.478427,0.028863,7
8,0.008758,0.002415,0.006331,0.000117,0.01,balanced,l2,liblinear,"{'C': 0.01, 'class_weight': 'balanced', 'penal...",0.480743,0.456059,0.523137,0.438818,0.492143,0.478180,0.029194,9
9,0.007008,0.001838,0.018144,0.014589,0.10,balanced,l1,liblinear,"{'C': 0.1, 'class_weight': 'balanced', 'penalt...",0.480743,0.456059,0.517222,0.448473,0.484601,0.477420,0.024253,10


In [40]:
model=grid_search_cv.best_estimator_

In [41]:
model.fit(xtrain,ytrain)

,penalty,'l1'
,dual,False
,tol,0.0001
,C,0.01
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [42]:
grid_search_cv.score(xtrain,ytrain)

0.5261900334261055

In [43]:
grid_search_cv.predict(xtest)

array(['No', 'No', 'Yes', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'Yes',
       'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No',
       'No', 'Yes', 'Yes', 'Yes', 'No', 'No', 'Yes', 'Yes', 'Yes', 'Yes',
       'No', 'Yes', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No',
       'No', 'Yes', 'No', 'Yes', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'No',
       'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'No', 'No', 'Yes',
       'No', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'No', 'Yes',
       'Yes', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes',
       'No', 'No', 'Yes', 'Yes', 'No', 'Yes', 'Yes', 'No', 'Yes', 'Yes',
       'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'No', 'No', 'No',
       'Yes', 'No', 'No', 'No', 'No', 'No', 'Yes', 'Yes', 'Yes', 'No',
       'Yes', 'Yes', 'Yes', 'No', 'No', 'No', 'No', 'Yes', 'No', 'Yes',
       'No', 'No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'No', 'Yes', 'Yes',
       'No', 'Yes', 'Yes', 'No', 'No', 'No', 'Yes', 'Yes', 'No

In [44]:
grid_search_cv.best_score_

np.float64(0.49363998388563346)

In [45]:
model.score(xtrain,ytrain)

0.52625